# 04 · Classical Baselines

**Goal:** reference AUCs for quark/gluon tagging with the same 3 inputs the quantum models will use
(`QG_mult`, `QG_ptD`, `QG_axis2`, the inputs of the CMS quark-gluon likelihood).

| Baseline | Role |
| --- | --- |
| CMS `jetQGl` | The experiment's own discriminant (physics reference, no training) |
| Gradient-boosted trees (BDT) | Standard HEP classifier: how much a classical model extracts from these inputs |
| RBF-kernel SVM | Classical twin of QSVC: same algorithm, classical kernel |

Each classical model is trained twice:
- on the **full training set** (imbalanced, C = 1, untuned; not an upper bound: the CV-tuned 400-jet RBF-SVM scores higher), and
- on **balanced 400-jet subsamples, 5 seeds** (the same size the quantum kernels will use; fair comparison).

All AUCs are on the same event-level test set as notebooks 02–03. Results are saved to `results/baselines.json`.

In [1]:
import sys
sys.path.insert(0, "src")

import json
from pathlib import Path
import numpy as np
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

from jet_data import load_jets, select_quark_gluon, feature_matrix, event_level_split, balanced_subsample, BOOKKEEPING

SEED = 42
SEEDS = [0, 1, 2, 3, 4]
N_PER_CLASS = 200                       # 400-jet balanced training subsamples
FEATURES = ("QG_mult", "QG_ptD", "QG_axis2")
RESULTS = Path("results"); RESULTS.mkdir(exist_ok=True)

raw = load_jets(features=FEATURES + ("jetQGl", "jetPt"), extra=BOOKKEEPING)
jets = select_quark_gluon(raw, require_loose_id=True)
y = jets["label"]
X = feature_matrix(jets, features=FEATURES)
train, test = event_level_split(jets, test_fraction=0.3, seed=SEED)
print(f"train {int(train.sum())}, test {int(test.sum())}, quark fraction (test) {y[test].mean():.3f}")

train 9729, test 4183, quark fraction (test) 0.372


## 1. CMS quark-gluon likelihood

`jetQGl = -1` means the likelihood could not be computed; those jets are excluded from this AUC only.

In [2]:
qgl = np.asarray(jets["jetQGl"], float)
valid = test & (qgl >= 0)
auc_qgl = roc_auc_score(y[valid], qgl[valid])
print(f"CMS QGL test AUC = {auc_qgl:.3f}   ({int((test & (qgl < 0)).sum())} test jets without QGL excluded)")

CMS QGL test AUC = 0.790   (2 test jets without QGL excluded)


## 2. Full training set

In [3]:
scaler = StandardScaler().fit(X[train])
Xtr, Xte = scaler.transform(X[train]), scaler.transform(X[test])

bdt = HistGradientBoostingClassifier(random_state=SEED).fit(Xtr, y[train])
auc_bdt_full = roc_auc_score(y[test], bdt.predict_proba(Xte)[:, 1])

svm = SVC(kernel="rbf", C=1.0, gamma="scale").fit(Xtr, y[train])
auc_svm_full = roc_auc_score(y[test], svm.decision_function(Xte))

print(f"BDT, full train:      test AUC = {auc_bdt_full:.3f}")
print(f"RBF-SVM, full train:  test AUC = {auc_svm_full:.3f}")

BDT, full train:      test AUC = 0.787
RBF-SVM, full train:  test AUC = 0.772


## 3. Balanced 400-jet subsamples (same size as the quantum experiment)

The scaler is fitted on each subsample only, exactly as the quantum pipeline will do.

In [4]:
sub = {"bdt": [], "rbf_svm": []}
for s in SEEDS:
    idx = balanced_subsample(y, train, N_PER_CLASS, seed=s)
    sc = StandardScaler().fit(X[idx])
    A, T = sc.transform(X[idx]), sc.transform(X[test])
    m = HistGradientBoostingClassifier(random_state=s).fit(A, y[idx])
    sub["bdt"].append(roc_auc_score(y[test], m.predict_proba(T)[:, 1]))
    m = SVC(kernel="rbf", C=1.0, gamma="scale").fit(A, y[idx])
    sub["rbf_svm"].append(roc_auc_score(y[test], m.decision_function(T)))

for name, vals in sub.items():
    print(f"{name:8s} 400 jets:  test AUC = {np.mean(vals):.3f} ± {np.std(vals):.3f}   ({', '.join(f'{v:.3f}' for v in vals)})")

bdt      400 jets:  test AUC = 0.746 ± 0.007   (0.736, 0.751, 0.738, 0.752, 0.751)
rbf_svm  400 jets:  test AUC = 0.782 ± 0.004   (0.777, 0.779, 0.781, 0.789, 0.784)


## 4. Dependence on jet pT

CMS builds its likelihood in bins of pT because the substructure variables depend strongly on pT, and the quark fraction changes with pT.
If AUCs are similar across bins, the inclusive result is not driven by the pT composition of the sample.

In [5]:
pt = np.asarray(jets["jetPt"], float)
edges = [30, 100, 300, np.inf]
print(f"{'pT bin [GeV]':16s} {'n test':>7s} {'quark frac':>10s} {'QGL AUC':>8s} {'BDT AUC':>8s}")
p_bdt = bdt.predict_proba(Xte)[:, 1]
for lo, hi in zip(edges[:-1], edges[1:]):
    in_bin = (pt >= lo) & (pt < hi)
    t = test & in_bin
    v = t & (qgl >= 0)
    tb = in_bin[test]
    label = f"{lo:.0f}-{hi:.0f}" if np.isfinite(hi) else f">{lo:.0f}"
    print(f"{label:16s} {int(t.sum()):7d} {y[t].mean():10.3f} "
          f"{roc_auc_score(y[v], qgl[v]):8.3f} {roc_auc_score(y[test][tb], p_bdt[tb]):8.3f}")

pT bin [GeV]      n test quark frac  QGL AUC  BDT AUC
30-100               779      0.184    0.740    0.716
100-300             1531      0.292    0.778    0.761
>300                1873      0.515    0.808    0.773


## 5. Save

In [6]:
baselines = {
    "features": list(FEATURES),
    "n_train_full": int(train.sum()), "n_test": int(test.sum()), "n_per_class_subsample": N_PER_CLASS,
    "seeds": SEEDS,
    "qgl_test_auc": auc_qgl,
    "bdt_full_auc": auc_bdt_full, "rbf_svm_full_auc": auc_svm_full,
    "bdt_sub_auc": sub["bdt"], "rbf_svm_sub_auc": sub["rbf_svm"],
}
json.dump(baselines, open(RESULTS / "baselines.json", "w"), indent=2)
print("saved results/baselines.json")
print()
print(f"- Baselines (3 QGL inputs, test {int(test.sum())} jets): CMS QGL {auc_qgl:.3f}; "
      f"BDT full {auc_bdt_full:.3f}, RBF-SVM full {auc_svm_full:.3f}; "
      f"400-jet subsamples: BDT {np.mean(sub['bdt']):.3f} ± {np.std(sub['bdt']):.3f}, "
      f"RBF-SVM {np.mean(sub['rbf_svm']):.3f} ± {np.std(sub['rbf_svm']):.3f}")

saved results/baselines.json

- Baselines (3 QGL inputs, test 4183 jets): CMS QGL 0.790; BDT full 0.787, RBF-SVM full 0.772; 400-jet subsamples: BDT 0.746 ± 0.007, RBF-SVM 0.782 ± 0.004
